# Treinamento do Modelo XGBoost

## Objetivo
Treinar modelo de classificação binária para prever movimentos de criptomoedas usando:
* **Dataset:** 305k registros com 52 features multi-timeframe
* **Target:** Adaptativo baseado em ATR (TP vs SL)
* **Balanceamento:** Class weights (scale_pos_weight=2.06)
* **Validação:** Split temporal (walk-forward)

## Estrutura
1. **Carregamento de dados**
2. **Preparação** (split temporal, features)
3. **Treinamento XGBoost** com class weights
4. **Avaliação** (AUC-ROC, Precision-Recall, F1, MCC)
5. **Feature importance**
6. **Salvamento do modelo**

## Datasets Disponíveis
* `default.crypto_features_with_adaptive_targets` - Dataset principal (305k registros)
* Target: `target_binary` (1=TP, 0=SL)
* Features: 52 (após remover metadata)
* Win rate: 32.7%
* Class weight: scale_pos_weight = 2.06

In [0]:
%pip install xgboost scikit-learn matplotlib seaborn

In [0]:
import pandas as pd
import numpy as np
from datetime import datetime
import matplotlib.pyplot as plt
import seaborn as sns

import xgboost as xgb
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import (
    roc_auc_score, roc_curve, auc,
    precision_recall_curve, average_precision_score,
    classification_report, confusion_matrix,
    f1_score, matthews_corrcoef
)

import pyspark.sql.functions as f
from pyspark.sql.window import Window

print(f"XGBoost {xgb.__version__}")
print(f"numPy {np.__version__}")
print(f"pandas {pd.__version__}")

## Carregamento dos Dados
Carregar o dataset preparado com:
* Target adaptativo baseado em ATR
* 52 features multi-timeframe (1h, 15m, 4h)
* Features de correlação (BTC, ETH benchmarks)
* 305k registros de 8 pares de criptomoedas

In [0]:
import pyspark.sql.functions as F

print("Carregando dataset com targets adaptativos...\n")

# Carregar tabela Delta
df_spark = spark.table("default.crypto_features_with_adaptive_targets")

print(f"Dataset carregado:")
print(f"  • Registros: {df_spark.count():,}")
print(f"  • Colunas: {len(df_spark.columns)}")
print(f"  • Pares: {df_spark.select('symbol').distinct().count()}")

# Mostrar período
period = df_spark.agg(F.min('timestamp'), F.max('timestamp')).collect()[0]
print(f"  • Período: {period[0]} a {period[1]}")

# Distribuição do target
print(f"\nDistribuição do target:")
target_dist = df_spark.groupBy('target_binary').count().orderBy('target_binary').collect()
for row in target_dist:
    pct = 100 * row['count'] / df_spark.count()
    print(f"  • Classe {row['target_binary']}: {row['count']:,} ({pct:.1f}%)")

In [0]:
print("Preparando features e target...\n")

# Definir colunas a excluir (metadata e target)
exclude_cols = [
    'timestamp', 'symbol', 'timeframe',  # Metadata
    'target_binary', 'target_horizon', 'target_return_atr', 'target_quality',  # Target
    'tp_threshold', 'sl_threshold', 'tp_reached', 'sl_reached',  # Intermediate
    'open', 'high', 'low', 'close', 'volume'  # OHLCV redundantes
]

# Selecionar apenas features + target + timestamp
feature_cols = [c for c in df_spark.columns if c not in exclude_cols]

print(f"Features selecionadas: {len(feature_cols)}")
print(f"   Excluídas: {len(exclude_cols)} colunas\n")

# Converter para Pandas (necessário para sklearn/xgboost)
print("Convertendo para Pandas...")

df_prep = df_spark.select(['timestamp', 'symbol'] + feature_cols + ['target_binary'])
df = df_prep.toPandas()

print(f"\nConversão completa!")
print(f"  • Shape: {df.shape}")
print(f"  • Memória: {df.memory_usage(deep=True).sum() / 1024**2:.1f} MB")
print(f"  • Features: {len(feature_cols)}")
print(f"  • Target: target_binary")

# Verificar nulos
nulls = df[feature_cols].isnull().sum().sum()
if nulls > 0:
    print(f"\nAtenção: {nulls:,} valores nulos encontrados")
    print("   Preenchendo com 0 (estratégia conservadora)...")
    df[feature_cols] = df[feature_cols].fillna(0)
    print("   Nulos preenchidos")
else:
    print(f"\nSem valores nulos!")

# Estatísticas básicas
print(f"\nEstatísticas do target:")
print(df['target_binary'].value_counts())
print(f"\nWin rate: {df['target_binary'].mean()*100:.1f}%")

## Split Temporal (Walk-Forward Validation)

### Por que split temporal?
Em séries temporais financeiras, **NÃO podemos usar split aleatório** pois:
* Cria data leakage (treina em dados futuros)
* Sobreavalia performance do modelo
* Não reflete uso real (sempre predizemos o futuro)

### Estratégia
* **Train:** 80% inicial (mais antigo)
* **Test:** 20% final (mais recente)
* **Split por tempo:** `shuffle=False`
* **Validação adicional:** Cross-validation temporal (TimeSeriesSplit)

In [0]:
print("Split temporal dos dados...\n")

# Ordenar por timestamp (garantir ordem temporal)
df = df.sort_values('timestamp').reset_index(drop=True)

print(f"Período total:")
print(f"  • Início: {df['timestamp'].min()}")
print(f"  • Fim: {df['timestamp'].max()}")
print(f"  • Duração: {(df['timestamp'].max() - df['timestamp'].min()).days} dias\n")

# Separar X e y
X = df[feature_cols]
y = df['target_binary']

# Split temporal 80/20 (SEM shuffle!)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    shuffle=False,  # IMPORTANTE: manter ordem temporal
    random_state=42
)

print(f"Split temporal completo:")
print(f"\n  TRAIN (80% - mais antigo):")
print(f"    • Registros: {len(X_train):,}")
print(f"    • Classe 0: {(y_train == 0).sum():,} ({100*(y_train == 0).sum()/len(y_train):.1f}%)")
print(f"    • Classe 1: {(y_train == 1).sum():,} ({100*(y_train == 1).sum()/len(y_train):.1f}%)")
print(f"    • Ratio: 1:{(y_train == 1).sum()/(y_train == 0).sum():.2f}")

print(f"\n  TEST (20% - mais recente):")
print(f"    • Registros: {len(X_test):,}")
print(f"    • Classe 0: {(y_test == 0).sum():,} ({100*(y_test == 0).sum()/len(y_test):.1f}%)")
print(f"    • Classe 1: {(y_test == 1).sum():,} ({100*(y_test == 1).sum()/len(y_test):.1f}%)")
print(f"    • Ratio: 1:{(y_test == 1).sum()/(y_test == 0).sum():.2f}")

## Treinamento do Modelo XGBoost

### Configurações
* **scale_pos_weight:** 2.06 (balanceamento via class weights)
* **max_depth:** 6 (controle de overfitting)
* **learning_rate:** 0.1 (padrão conservador)
* **n_estimators:** 200 (árvores)
* **objective:** binary:logistic (classificação binária)
* **eval_metric:** auc (AUC-ROC)
* **early_stopping:** 20 rounds (evita overfitting)

### Por que scale_pos_weight=2.06?
* Ratio de desbalanceamento: 1:2.06 (SL vs TP)
* Penaliza 2x mais erros na classe minoritária
* Equivalente a SMOTE, mas mais eficiente

In [0]:
print("Treinando com XGBoost e class weights...\n")

# Calcular scale_pos_weight (ratio de desbalanceamento)
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
print(f"Class weight calculado: {scale_pos_weight:.4f}\n")

# Configurar modelo XGBoost
model = xgb.XGBClassifier(
    scale_pos_weight=scale_pos_weight,  # Balanceamento
    max_depth=6,                         # Profundidade da árvore
    learning_rate=0.1,                   # Taxa de aprendizado
    n_estimators=200,                    # Número de árvores
    subsample=0.8,                       # Amostragem de linhas
    colsample_bytree=0.8,                # Amostragem de colunas
    objective='binary:logistic',         # Classificação binária
    eval_metric='auc',                   # Métrica de avaliação
    early_stopping_rounds=20,            # Early stopping
    random_state=42,
    n_jobs=-1,                           # Usar todos os cores
    verbosity=1
)

print("Parâmetros do modelo:")
for key, value in model.get_params().items():
    if key in ['scale_pos_weight', 'max_depth', 'learning_rate', 'n_estimators', 'subsample', 'colsample_bytree', 'early_stopping_rounds']:
        print(f"  • {key}: {value}")

print(f"\nIniciando treinamento...")
print(f"   (com early stopping em 20 rounds)\n")

# Treinar com early stopping
model.fit(
    X_train, y_train,
    eval_set=[(X_train, y_train), (X_test, y_test)],
    verbose=False  # Reduzir output
)

print(f"\nTreinamento completo!")
print(f"  • Melhor iteração: {model.best_iteration}")
print(f"  • Melhor score: {model.best_score:.4f}")

## Avaliação do Modelo

### Métricas Principais
1. **AUC-ROC** - Capacidade de discriminação (alvo: > 65%)
2. **Precision-Recall AUC** - Desempenho na classe minoritária
3. **F1-Score** - Balanço entre precisão e recall
4. **MCC** - Matthews Correlation Coefficient (robusto para desbalanceamento)

### Interpretação
* **AUC-ROC:**
  * 0.50 = Aleatório (inútil)
  * 0.60-0.70 = Razoável
  * 0.70-0.80 = Bom
  * 0.80+ = Excelente

* **F1-Score:**
  * 0.0-0.4 = Fraco
  * 0.4-0.6 = Moderado
  * 0.6-0.8 = Bom
  * 0.8+ = Excelente

In [0]:
# Predições
y_pred_proba = model.predict_proba(X_test)[:, 1]  # Probabilidades
y_pred = model.predict(X_test)                     # Classes (0 ou 1)

print("="*70)
print("RESULTADOS DA AVALIAÇÃO")
print("="*70)

# 1. AUC-ROC
auc_roc = roc_auc_score(y_test, y_pred_proba)
print(f"\nAUC-ROC: {auc_roc:.4f}")
if auc_roc >= 0.75:
    print("   EXCELENTE. Modelo com alta capacidade preditiva.")
elif auc_roc >= 0.65:
    print("   BOM. Modelo com capacidade preditiva satisfatória.")
elif auc_roc >= 0.55:
    print("   RAZOÁVEL. Modelo tem algum poder preditivo, mas pode melhorar.")
else:
    print("   FRACO. Modelo próximo ao aleatório.")

# 2. Precision-Recall AUC
pr_auc = average_precision_score(y_test, y_pred_proba)
print(f"\nPrecision-Recall AUC: {pr_auc:.4f}")
print(f"   (Desempenho na classe minoritária - TP)")

# 3. F1-Score
f1 = f1_score(y_test, y_pred)
print(f"\nF1-Score: {f1:.4f}")
print(f"   (Balanço entre precisão e recall)")

# 4. Matthews Correlation Coefficient
mcc = matthews_corrcoef(y_test, y_pred)
print(f"\nMatthews Correlation Coefficient (MCC): {mcc:.4f}")
print(f"   (Correlação entre predições e realidade)")

# 5. Classification Report
print(f"\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['SL (0)', 'TP (1)']))

# 6. Confusion Matrix
print(f"\nConfusion Matrix:")
cm = confusion_matrix(y_test, y_pred)
print(f"\n{cm}\n")
print(f"   Verdadeiros Negativos (TN): {cm[0,0]:,} - Corretamente previu SL")
print(f"   Falsos Positivos (FP): {cm[0,1]:,} - Previu TP mas era SL")
print(f"   Falsos Negativos (FN): {cm[1,0]:,} - Previu SL mas era TP")
print(f"   Verdadeiros Positivos (TP): {cm[1,1]:,} - Corretamente previu TP")

# Calcular métricas derivadas
accuracy = (cm[0,0] + cm[1,1]) / cm.sum()
precision_tp = cm[1,1] / (cm[0,1] + cm[1,1]) if (cm[0,1] + cm[1,1]) > 0 else 0
recall_tp = cm[1,1] / (cm[1,0] + cm[1,1]) if (cm[1,0] + cm[1,1]) > 0 else 0

print(f"\nMétricas Derivadas:")
print(f"  • Acurácia: {accuracy:.4f} (usar com cautela em dados desbalanceados)")
print(f"  • Precisão (TP): {precision_tp:.4f} (quando prevê TP, quantos são corretos)")
print(f"  • Recall (TP): {recall_tp:.4f} (dos TPs reais, quantos identificou)")

print("\n" + "="*70)

# Resumo final
print(f"\nRESUMO FINAL:")
print(f"  • AUC-ROC: {auc_roc:.4f} ({100*auc_roc:.1f}%)")
print(f"  • F1-Score: {f1:.4f}")
print(f"  • MCC: {mcc:.4f}")
print(f"  • Win rate test: {y_test.mean()*100:.1f}%")

if auc_roc >= 0.65:
    print(f"\nSucesso! Modelo superou baseline aleatório (0.52) significativamente!")
else:
    print(f"\nModelo precisa de melhorias. Considerar:")
    print("  1. Feature engineering adicional")
    print("  2. Otimização de hiperparâmetros")
    print("  3. Ensemble de modelos")
    print("  4. Análise de features mais importantes")

In [0]:
# Configurar estilo
plt.style.use('default')
sns.set_palette("husl")

# Criar figura com subplots
fig, axes = plt.subplots(2, 2, figsize=(15, 12))
fig.suptitle('Avaliação do Modelo XGBoost', fontsize=16, fontweight='bold')

# 1. ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_pred_proba)
roc_auc = auc(fpr, tpr)

axes[0, 0].plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc:.4f})')
axes[0, 0].plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random (AUC = 0.50)')
axes[0, 0].set_xlim([0.0, 1.0])
axes[0, 0].set_ylim([0.0, 1.05])
axes[0, 0].set_xlabel('False Positive Rate')
axes[0, 0].set_ylabel('True Positive Rate')
axes[0, 0].set_title('ROC Curve')
axes[0, 0].legend(loc="lower right")
axes[0, 0].grid(alpha=0.3)

# 2. Precision-Recall Curve
precision, recall, _ = precision_recall_curve(y_test, y_pred_proba)
pr_auc = average_precision_score(y_test, y_pred_proba)

axes[0, 1].plot(recall, precision, color='green', lw=2, label=f'PR curve (AUC = {pr_auc:.4f})')
baseline = y_test.mean()
axes[0, 1].axhline(y=baseline, color='navy', linestyle='--', lw=2, label=f'Baseline ({baseline:.2f})')
axes[0, 1].set_xlim([0.0, 1.0])
axes[0, 1].set_ylim([0.0, 1.05])
axes[0, 1].set_xlabel('Recall')
axes[0, 1].set_ylabel('Precision')
axes[0, 1].set_title('Precision-Recall Curve')
axes[0, 1].legend(loc="best")
axes[0, 1].grid(alpha=0.3)

# 3. Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[1, 0],
            xticklabels=['SL (0)', 'TP (1)'],
            yticklabels=['SL (0)', 'TP (1)'])
axes[1, 0].set_ylabel('True Label')
axes[1, 0].set_xlabel('Predicted Label')
axes[1, 0].set_title('Confusion Matrix')

# 4. Feature Importance (Top 20)
feature_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False).head(20)

axes[1, 1].barh(range(len(feature_importance)), feature_importance['importance'])
axes[1, 1].set_yticks(range(len(feature_importance)))
axes[1, 1].set_yticklabels(feature_importance['feature'])
axes[1, 1].invert_yaxis()
axes[1, 1].set_xlabel('Importance')
axes[1, 1].set_title('Top 20 Features mais Importantes')
axes[1, 1].grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

In [0]:
# Criar DataFrame com importances
feature_importance_df = pd.DataFrame({
    'feature': feature_cols,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

print("Top 20 Features mais Importantes:\n")
print(feature_importance_df.head(20).to_string(index=False))

# Análise por categoria de feature
print("\n\nImportância por Categoria:\n")

categories = {
    '1h': [f for f in feature_cols if '_1h' in f],
    '15m': [f for f in feature_cols if '_15m' in f],
    '4h': [f for f in feature_cols if '_4h' in f],
    'BTC benchmark': [f for f in feature_cols if 'btc_' in f or 'beta_btc' in f],
    'ETH benchmark': [f for f in feature_cols if 'eth_' in f],
    'Correlação': [f for f in feature_cols if 'relative_strength' in f or 'ratio' in f]
}

for cat_name, cat_features in categories.items():
    cat_importance = feature_importance_df[feature_importance_df['feature'].isin(cat_features)]['importance'].sum()
    cat_pct = 100 * cat_importance / feature_importance_df['importance'].sum()
    print(f"  {cat_name:20s}: {cat_importance:.4f} ({cat_pct:.1f}%)")

print(f"\nInsights:")
top_feature = feature_importance_df.iloc[0]['feature']
top_importance = feature_importance_df.iloc[0]['importance']
print(f"  • Feature mais importante: {top_feature} ({top_importance:.4f})")

top_5_importance = feature_importance_df.head(5)['importance'].sum()
total_importance = feature_importance_df['importance'].sum()
print(f"  • Top 5 features respondem por {100*top_5_importance/total_importance:.1f}% da importância")

if any('_1h' in f for f in feature_importance_df.head(10)['feature'].values):
    print(f"  • Features de 1h são dominantes (escala operacional ideal)")
if any('btc_' in f or 'beta_btc' in f for f in feature_importance_df.head(10)['feature'].values):
    print(f"  • Correlação com BTC é relevante (mercado segue Bitcoin)")

## Save model

Salvar modelo treinado para:
* Uso em produção
* Comparação com futuros modelos
* Reprodução de resultados

### Formato
* **XGBoost nativo:** `.json` (recomendado, mais portável)
* **Pickle:** `.pkl` (alternativa)

### Metadados Salvos
* Performance (AUC, F1, MCC)
* Configurações do modelo
* Lista de features
* Data do treino

In [0]:
import json
import pickle
from datetime import datetime

print("\nSalvando modelo treinado...\n")

# Criar diretório se não existir
import os
model_dir = os.path.join(os.path.dirname(os.getcwd()), 'models')
os.makedirs(model_dir, exist_ok=True)

# Nome do arquivo
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
# model_name = f"xgboost_atr_target_{timestamp}"
model_name = f"xgboost_atr_target"

# 1. Salvar modelo XGBoost (formato JSON - recomendado)
model_path_json = f"{model_dir}/{model_name}.json"
model.save_model(model_path_json)
print(f"Modelo salvo (JSON): {model_path_json}")

# 2. Salvar modelo Pickle (alternativa)
model_path_pkl = f"{model_dir}/{model_name}.pkl"
with open(model_path_pkl, 'wb') as f:
    pickle.dump(model, f)
print(f"Modelo salvo (Pickle): {model_path_pkl}")

# 3. Salvar metadados
metadata = {
    'model_name': model_name,
    'model_type': 'XGBClassifier',
    'training_date': datetime.now().isoformat(),
    'dataset': 'default.crypto_features_with_adaptive_targets',
    'n_features': len(feature_cols),
    'features': feature_cols,
    'n_train': len(X_train),
    'n_test': len(X_test),
    'performance': {
        'auc_roc': float(auc_roc),
        'pr_auc': float(pr_auc),
        'f1_score': float(f1),
        'mcc': float(mcc),
        'accuracy': float(accuracy),
        'precision_tp': float(precision_tp),
        'recall_tp': float(recall_tp)
    },
    'model_params': model.get_params(),
    'class_balance': {
        'train_ratio': float((y_train == 1).sum() / (y_train == 0).sum()),
        'test_ratio': float((y_test == 1).sum() / (y_test == 0).sum()),
        'scale_pos_weight': float(scale_pos_weight)
    },
    'best_iteration': int(model.best_iteration),
    'best_score': float(model.best_score)
}

metadata_path = f"{model_dir}/{model_name}_metadata.json"
with open(metadata_path, 'w') as f:
    json.dump(metadata, f, indent=2, default=str)
print(f"Metadados salvos: {metadata_path}")

# 4. Salvar feature importance
feature_importance_path = f"{model_dir}/{model_name}_feature_importance.csv"
feature_importance_df.to_csv(feature_importance_path, index=False)
print(f"Feature importance salvo: {feature_importance_path}")

print(f"\nArquivos criados:")
print(f"  1. {model_name}.json (modelo XGBoost)")
print(f"  2. {model_name}.pkl (modelo Pickle)")
print(f"  3. {model_name}_metadata.json (metadados)")
print(f"  4. {model_name}_feature_importance.csv (importâncias)")

# Resumo Final do Treinamento

## Dataset
* **Registros:** 305k (train: 244k, test: 61k)
* **Features:** 52 multi-timeframe (1h, 15m, 4h)
* **Target:** Adaptativo baseado em ATR (TP vs SL)
* **Balanceamento:** Class weights (scale_pos_weight)

## Modelo
* **Algoritmo:** XGBoost
* **Hiperparâmetros:** max_depth=6, learning_rate=0.1, n_estimators=200
* **Balanceamento:** scale_pos_weight calculado automaticamente
* **Early stopping:** 20 rounds

## Performance
* **AUC-ROC:** (ver resultados acima)
* **F1-Score:** (ver resultados acima)
* **MCC:** (ver resultados acima)
* **Win rate test:** ~33%

## Próximos Passos

### Prioridade ALTA
1. ✅ **Treinar modelo XGBoost** - COMPLETO
2. 🔴 **Otimizar hiperparâmetros**
   - Grid search ou Bayesian optimization
   - Testar diferentes valores de max_depth, learning_rate, etc.
3. 🔴 **Testar outros algoritmos**
   - LightGBM
   - CatBoost
   - Ensemble (voting/stacking)

### Prioridade MÉDIA
4. 🟡 **Walk-forward validation**
   - Simulação temporal mais robusta
   - Avaliar por regime de mercado
5. 🟡 **Feature engineering adicional**
   - Sentiment (funding rate, open interest)
   - Order book features
6. 🟡 **Integração com bot**
   - Usar modelo em produção
   - Backtest realista

### Prioridade BAIXA
7. 🟢 Explainability (SHAP values)
8. 🟢 Monitoring e retraining automático